Set up packages, Colab paths, data folders, and result folders.


In [1]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TQDM_DISABLE"] = "1"


def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


for import_name, pip_name in [
    ("thop", "thop"),
    ("torchinfo", "torchinfo"),
    ("transformers", "transformers"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("tqdm", "tqdm"),
]:
    ensure_package(import_name, pip_name)

try:
    from huggingface_hub.utils import disable_progress_bars

    disable_progress_bars()
except Exception:
    pass


def project_paths(problem_name):
    here = Path.cwd()
    drive_root = Path("/content/drive/MyDrive")

    if Path("/content").exists() and not drive_root.exists():
        try:
            from google.colab import drive

            drive.mount("/content/drive")
        except Exception:
            pass

    if drive_root.exists():
        work_dir = drive_root / "homework_5"
    elif Path("/content").exists():
        work_dir = Path("/content/homework_5")
    else:
        work_dir = here / "homework_5"

    result_dir = work_dir / "results" / problem_name
    data_dir = work_dir / "data"
    result_dir.mkdir(parents=True, exist_ok=True)
    data_dir.mkdir(parents=True, exist_ok=True)
    return work_dir, data_dir, result_dir


Define shared helpers for reproducibility, device selection, CIFAR-100 loading, metrics, exports, plots, and training loops.


In [2]:
"""Reproducibility and runtime-device helpers."""


import os
import random
from dataclasses import dataclass

import numpy as np
import torch


@dataclass(frozen=True)
class DeviceInfo:
    """Small serializable description of the selected torch device."""

    device: torch.device
    name: str
    cuda_available: bool
    cuda_device_count: int

    def as_dict(self) -> dict[str, object]:
        return {
            "device": str(self.device),
            "name": self.name,
            "cuda_available": self.cuda_available,
            "cuda_device_count": self.cuda_device_count,
        }


def is_colab() -> bool:
    """Return True when running inside a Google Colab runtime."""

    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in os.environ


def set_seed(seed: int = 42, deterministic: bool = False) -> None:
    """Set practical random seeds for Python, NumPy, and PyTorch."""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = True


def select_device(prefer_cuda: bool = True) -> DeviceInfo:
    """Select CUDA when available, otherwise CPU."""

    cuda_available = bool(torch.cuda.is_available())
    if prefer_cuda and cuda_available:
        device = torch.device("cuda")
        name = torch.cuda.get_device_name(device)
    else:
        device = torch.device("cpu")
        name = "CPU"
    return DeviceInfo(
        device=device,
        name=name,
        cuda_available=cuda_available,
        cuda_device_count=torch.cuda.device_count(),
    )


def cuda_synchronize_if_needed(device: torch.device | str) -> None:
    """Synchronize CUDA timings only when the active device is CUDA."""

    device = torch.device(device)
    if device.type == "cuda":
        torch.cuda.synchronize(device)


"""Metrics, complexity estimates, plotting, and structured exports."""


import csv
import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import torch


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def count_parameters(model: torch.nn.Module) -> tuple[int, int]:
    """Return total and trainable parameter counts."""

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def accuracy_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    """Compute top-1 accuracy for a batch of raw logits."""

    predictions = logits.argmax(dim=1)
    return (predictions == labels).float().mean().item()


def model_size_megabytes(model: torch.nn.Module) -> float:
    """Estimate parameter storage size in MiB."""

    bytes_total = sum(p.numel() * p.element_size() for p in model.parameters())
    return bytes_total / (1024**2)


def estimate_macs_thop(
    model: torch.nn.Module,
    input_size: tuple[int, int, int, int],
    device: torch.device | str,
) -> dict[str, Any]:
    """Estimate MACs with thop and label the convention explicitly."""

    try:
        from thop import profile
    except Exception as exc:  # pragma: no cover - optional dependency
        return {
            "complexity_tool": "thop",
            "complexity_status": "unavailable",
            "macs": None,
            "flops_convention": "not_computed",
            "message": str(exc),
        }

    was_training = model.training
    model.eval()
    dummy = torch.randn(*input_size, device=device)
    try:
        macs, params = profile(model, inputs=(dummy,), verbose=False)
        return {
            "complexity_tool": "thop",
            "complexity_status": "ok",
            "macs": int(macs),
            "params_seen_by_tool": int(params),
            "flops_convention": "thop returns MACs; FLOPs are often approximated as 2x MACs",
            "estimated_flops_if_2x_macs": int(2 * macs),
        }
    except Exception as exc:  # pragma: no cover - model/operator dependent
        return {
            "complexity_tool": "thop",
            "complexity_status": "failed",
            "macs": None,
            "flops_convention": "not_computed",
            "message": str(exc),
        }
    finally:
        if was_training:
            model.train()


def save_json(data: Any, path: str | Path) -> Path:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    return path


def save_csv(rows: list[dict[str, Any]], path: str | Path) -> Path:
    path = Path(path)
    ensure_dir(path.parent)
    if not rows:
        raise ValueError("empty csv")
    fieldnames: list[str] = sorted({key for row in rows for key in row})
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path


def save_training_history(history: list[dict[str, Any]], path: str | Path) -> Path:
    return save_csv(history, path)


def plot_training_history(
    history: list[dict[str, Any]],
    title: str,
    path: str | Path,
) -> Path:
    """Save loss and accuracy curves from a per-epoch history table."""

    path = Path(path)
    ensure_dir(path.parent)
    epochs = [row["epoch"] for row in history]
    train_loss = [row.get("train_loss") for row in history]
    val_acc = [row.get("eval_accuracy") for row in history]

    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(epochs, train_loss, marker="o", label="Train loss", color="#1f77b4")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Training loss")
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(epochs, val_acc, marker="s", label="Eval accuracy", color="#2ca02c")
    ax2.set_ylabel("Eval accuracy")

    lines = ax1.get_lines() + ax2.get_lines()
    labels = [line.get_label() for line in lines]
    ax1.legend(lines, labels, loc="best")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


"""CIFAR-100 data loading and preprocessing helpers."""


from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from torchvision import datasets, transforms


CIFAR100_NUM_CLASSES = 100
CIFAR100_IMAGE_SIZE = 32
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD = (0.2675, 0.2565, 0.2761)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def build_cifar100_transforms(
    image_size: int,
    train: bool,
    mean: tuple[float, float, float] = CIFAR100_MEAN,
    std: tuple[float, float, float] = CIFAR100_STD,
    augment: bool = True,
) -> transforms.Compose:
    """Create transforms for CIFAR-100 at either native or resized resolution."""

    steps: list[object] = []
    if image_size != CIFAR100_IMAGE_SIZE:
        steps.append(transforms.Resize((image_size, image_size), antialias=True))
    if train and augment:
        if image_size == CIFAR100_IMAGE_SIZE:
            steps.extend(
                [
                    transforms.RandomCrop(CIFAR100_IMAGE_SIZE, padding=4),
                    transforms.RandomHorizontalFlip(),
                ]
            )
        else:
            steps.append(transforms.RandomHorizontalFlip())
    steps.extend([transforms.ToTensor(), transforms.Normalize(mean, std)])
    return transforms.Compose(steps)


def _limit_dataset(dataset: Dataset, max_items: int | None) -> Dataset:
    if max_items is None:
        return dataset
    return Subset(dataset, list(range(min(max_items, len(dataset)))))


def get_cifar100_dataloaders(
    data_root: str | Path,
    batch_size: int,
    image_size: int,
    mean: tuple[float, float, float] = CIFAR100_MEAN,
    std: tuple[float, float, float] = CIFAR100_STD,
    augment_train: bool = True,
    num_workers: int = 2,
    pin_memory: bool | None = None,
    validation_size: int = 0,
    seed: int = 42,
    max_train_items: int | None = None,
    max_test_items: int | None = None,
    download: bool = True,
) -> dict[str, DataLoader]:
    """Download CIFAR-100 and return train/test or train/val/test loaders."""

    data_root = Path(data_root)
    train_transform = build_cifar100_transforms(image_size, True, mean, std, augment_train)
    test_transform = build_cifar100_transforms(image_size, False, mean, std, False)

    train_dataset = datasets.CIFAR100(
        root=data_root,
        train=True,
        download=download,
        transform=train_transform,
    )
    test_dataset = datasets.CIFAR100(
        root=data_root,
        train=False,
        download=download,
        transform=test_transform,
    )

    if validation_size > 0:
        if validation_size >= len(train_dataset):
            raise ValueError("bad validation_size")
        train_size = len(train_dataset) - validation_size
        generator = torch.Generator().manual_seed(seed)
        train_dataset, val_dataset = random_split(
            train_dataset,
            [train_size, validation_size],
            generator=generator,
        )
    else:
        val_dataset = None

    train_dataset = _limit_dataset(train_dataset, max_train_items)
    test_dataset = _limit_dataset(test_dataset, max_test_items)
    if val_dataset is not None:
        val_dataset = _limit_dataset(val_dataset, max_test_items)

    if pin_memory is None:
        pin_memory = torch.cuda.is_available()

    loader_kwargs = {
        "batch_size": batch_size,
        "num_workers": num_workers,
        "pin_memory": pin_memory,
    }
    loaders = {
        "train": DataLoader(train_dataset, shuffle=True, **loader_kwargs),
        "test": DataLoader(test_dataset, shuffle=False, **loader_kwargs),
    }
    if val_dataset is not None:
        loaders["val"] = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
    return loaders


def validate_cifar100_labels(loader: DataLoader) -> tuple[int, int]:
    """Return min/max labels from one loader and assert CIFAR-100 range."""

    min_label = CIFAR100_NUM_CLASSES
    max_label = -1
    for _, labels in loader:
        min_label = min(min_label, int(labels.min().item()))
        max_label = max(max_label, int(labels.max().item()))
    if min_label < 0 or max_label >= CIFAR100_NUM_CLASSES:
        raise ValueError(f"bad labels: {min_label}-{max_label}")
    return min_label, max_label


"""Reusable supervised training and evaluation loops."""


import time
from pathlib import Path
from typing import Iterable

import torch
from torch import nn



def extract_logits(model_output):
    """Handle plain Tensor outputs and Hugging Face outputs with .logits."""

    if hasattr(model_output, "logits"):
        return model_output.logits
    if isinstance(model_output, tuple):
        return model_output[0]
    return model_output


def set_named_modules_eval(model: nn.Module, module_names: Iterable[str]) -> None:
    """Put selected top-level modules into eval mode during head-only training."""

    for name in module_names:
        module = getattr(model, name, None)
        if isinstance(module, nn.Module):
            module.eval()


def train_one_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device | str,
    max_batches: int | None = None,
    frozen_eval_modules: Iterable[str] = (),
) -> dict[str, float]:
    """Train for one epoch and return loss, accuracy, and elapsed seconds."""

    device = torch.device(device)
    model.train()
    set_named_modules_eval(model, frozen_eval_modules)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    cuda_synchronize_if_needed(device)
    start = time.perf_counter()

    for batch_idx, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = extract_logits(model(images))
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

    cuda_synchronize_if_needed(device)
    elapsed = time.perf_counter() - start
    if total_examples == 0:
        raise RuntimeError("no train batches")
    return {
        "train_loss": total_loss / total_examples,
        "train_accuracy": total_correct / total_examples,
        "epoch_time_seconds": elapsed,
    }


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    device: torch.device | str,
    max_batches: int | None = None,
) -> dict[str, float]:
    """Evaluate with gradients disabled."""

    device = torch.device(device)
    was_training = model.training
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for batch_idx, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = extract_logits(model(images))
        loss = criterion(logits, labels)
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size
    if was_training:
        model.train()
    if total_examples == 0:
        raise RuntimeError("no eval batches")
    return {
        "eval_loss": total_loss / total_examples,
        "eval_accuracy": total_correct / total_examples,
    }


def save_checkpoint(
    model: nn.Module,
    path: str | Path,
    metadata: dict | None = None,
) -> Path:
    """Save model weights plus serializable metadata."""

    path = Path(path)
    ensure_dir(path.parent)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "metadata": metadata or {},
        },
        path,
    )
    return path


def fit_classifier(
    model: nn.Module,
    train_loader,
    eval_loader,
    optimizer: torch.optim.Optimizer,
    device: torch.device | str,
    epochs: int,
    checkpoint_path: str | Path | None = None,
    max_train_batches: int | None = None,
    max_eval_batches: int | None = None,
    frozen_eval_modules: Iterable[str] = (),
) -> tuple[list[dict], dict]:
    """Train/evaluate a classifier and return per-epoch history plus summary."""

    criterion = nn.CrossEntropyLoss()
    history: list[dict] = []
    best_accuracy = -1.0
    total_start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
            max_batches=max_train_batches,
            frozen_eval_modules=frozen_eval_modules,
        )
        eval_metrics = evaluate(
            model,
            eval_loader,
            criterion,
            device,
            max_batches=max_eval_batches,
        )
        row = {"epoch": epoch, **train_metrics, **eval_metrics}
        history.append(row)
        if checkpoint_path is not None and eval_metrics["eval_accuracy"] >= best_accuracy:
            best_accuracy = eval_metrics["eval_accuracy"]
            save_checkpoint(
                model,
                checkpoint_path,
                {"best_epoch": epoch, "best_eval_accuracy": best_accuracy},
            )

    total_time = time.perf_counter() - total_start
    summary = {
        "total_training_time_seconds": total_time,
        "mean_epoch_time_seconds": sum(r["epoch_time_seconds"] for r in history) / len(history),
        "final_eval_accuracy": history[-1]["eval_accuracy"],
        "final_eval_loss": history[-1]["eval_loss"],
    }
    return history, summary


def verify_optimizer_parameters(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
) -> tuple[int, int]:
    """Return trainable parameter count and optimizer parameter count."""

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    optimized = 0
    seen: set[int] = set()
    for group in optimizer.param_groups:
        for param in group["params"]:
            if id(param) not in seen:
                optimized += param.numel()
                seen.add(id(param))
    if trainable != optimized:
        raise ValueError("bad optimizer params")
    return trainable, optimized


Define the compact scratch Swin Transformer architecture and its shape checks.


In [3]:
"""Compact Swin Transformer from scratch for CIFAR-100 experiments."""


from dataclasses import dataclass

import torch
from torch import nn


def window_partition(x: torch.Tensor, window_size: int) -> torch.Tensor:
    """Partition BHWC feature maps into non-overlapping windows."""

    bsz, height, width, channels = x.shape
    if height % window_size != 0 or width % window_size != 0:
        raise ValueError("bad window_size")
    x = x.view(
        bsz,
        height // window_size,
        window_size,
        width // window_size,
        window_size,
        channels,
    )
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return windows.view(-1, window_size, window_size, channels)


def window_reverse(
    windows: torch.Tensor,
    window_size: int,
    height: int,
    width: int,
) -> torch.Tensor:
    """Reverse window_partition for BHWC feature maps."""

    bsz = int(windows.shape[0] / (height * width / window_size / window_size))
    x = windows.view(
        bsz,
        height // window_size,
        width // window_size,
        window_size,
        window_size,
        -1,
    )
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return x.view(bsz, height, width, -1)


class DropPath(nn.Module):
    """Stochastic depth; identity when drop_prob is zero."""

    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor


class PatchEmbed(nn.Module):
    """Patch embedding for Swin-style hierarchical features."""

    def __init__(self, image_size: int = 224, patch_size: int = 4, in_channels: int = 3, embed_dim: int = 48):
        super().__init__()
        if image_size % patch_size != 0:
            raise ValueError("bad patch_size")
        self.image_size = image_size
        self.patch_size = patch_size
        self.grid_size = image_size // patch_size
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, int, int]:
        x = self.proj(x)
        height, width = x.shape[-2:]
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x, height, width


class WindowAttention(nn.Module):
    """Window multi-head self-attention with relative position bias."""

    def __init__(self, dim: int, window_size: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        if dim % num_heads != 0:
            raise ValueError("bad heads")
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim**-0.5
        table_size = (2 * window_size - 1) * (2 * window_size - 1)
        self.relative_position_bias_table = nn.Parameter(torch.zeros(table_size, num_heads))

        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing="ij"))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)

        self.qkv = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(dropout)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(dropout)
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        b_windows, tokens, channels = x.shape
        qkv = self.qkv(x).reshape(b_windows, tokens, 3, self.num_heads, channels // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        relative_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(tokens, tokens, -1)
        relative_bias = relative_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_bias.unsqueeze(0)

        if mask is not None:
            num_windows = mask.shape[0]
            attn = attn.view(b_windows // num_windows, num_windows, self.num_heads, tokens, tokens)
            attn = attn + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, tokens, tokens)

        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(b_windows, tokens, channels)
        x = self.proj(x)
        return self.proj_drop(x)


class SwinBlock(nn.Module):
    """Swin Transformer block with regular or shifted-window attention."""

    def __init__(
        self,
        dim: int,
        input_resolution: tuple[int, int],
        num_heads: int,
        window_size: int = 7,
        shift_size: int = 0,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
        drop_path: float = 0.0,
    ):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.window_size = min(window_size, input_resolution[0], input_resolution[1])
        self.shift_size = 0 if min(input_resolution) <= window_size else shift_size
        if self.shift_size >= self.window_size:
            raise ValueError("bad shift_size")

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, self.window_size, num_heads, dropout)
        self.drop_path = DropPath(drop_path)
        self.norm2 = nn.LayerNorm(dim)
        hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )
        self.register_buffer("attn_mask", self._create_mask(), persistent=False)

    def _create_mask(self) -> torch.Tensor | None:
        height, width = self.input_resolution
        if self.shift_size == 0:
            return None
        img_mask = torch.zeros((1, height, width, 1))
        h_slices = (
            slice(0, -self.window_size),
            slice(-self.window_size, -self.shift_size),
            slice(-self.shift_size, None),
        )
        w_slices = (
            slice(0, -self.window_size),
            slice(-self.window_size, -self.shift_size),
            slice(-self.shift_size, None),
        )
        count = 0
        for h_slice in h_slices:
            for w_slice in w_slices:
                img_mask[:, h_slice, w_slice, :] = count
                count += 1
        mask_windows = window_partition(img_mask, self.window_size).view(-1, self.window_size * self.window_size)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0))
        attn_mask = attn_mask.masked_fill(attn_mask == 0, float(0.0))
        return attn_mask

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        height, width = self.input_resolution
        bsz, length, channels = x.shape
        if length != height * width:
            raise ValueError("bad token_count")
        shortcut = x
        x = self.norm1(x).view(bsz, height, width, channels)
        if self.shift_size > 0:
            shifted = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted = x

        windows = window_partition(shifted, self.window_size)
        windows = windows.view(-1, self.window_size * self.window_size, channels)
        attn_windows = self.attn(windows, mask=self.attn_mask)
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, channels)
        shifted = window_reverse(attn_windows, self.window_size, height, width)

        if self.shift_size > 0:
            x = torch.roll(shifted, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted
        x = x.view(bsz, height * width, channels)
        x = shortcut + self.drop_path(x)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class PatchMerging(nn.Module):
    """Downsample by merging each 2x2 neighborhood and doubling channels."""

    def __init__(self, input_resolution: tuple[int, int], dim: int):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = nn.LayerNorm(4 * dim)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, int, int]:
        height, width = self.input_resolution
        bsz, length, channels = x.shape
        if length != height * width:
            raise ValueError("bad token_count")
        if height % 2 != 0 or width % 2 != 0:
            raise ValueError("bad merge_size")
        x = x.view(bsz, height, width, channels)
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], dim=-1)
        x = x.view(bsz, -1, 4 * channels)
        x = self.norm(x)
        x = self.reduction(x)
        return x, height // 2, width // 2


class BasicLayer(nn.Module):
    """A Swin stage made of alternating W-MSA and SW-MSA blocks."""

    def __init__(
        self,
        dim: int,
        input_resolution: tuple[int, int],
        depth: int,
        num_heads: int,
        window_size: int,
        dropout: float,
        downsample: bool,
    ):
        super().__init__()
        self.blocks = nn.ModuleList(
            [
                SwinBlock(
                    dim=dim,
                    input_resolution=input_resolution,
                    num_heads=num_heads,
                    window_size=window_size,
                    shift_size=0 if i % 2 == 0 else window_size // 2,
                    dropout=dropout,
                )
                for i in range(depth)
            ]
        )
        self.downsample = PatchMerging(input_resolution, dim) if downsample else None

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, int, int]:
        for block in self.blocks:
            x = block(x)
        if self.downsample is not None:
            x, height, width = self.downsample(x)
        else:
            height, width = self.blocks[-1].input_resolution
        return x, height, width


@dataclass(frozen=True)
class ScratchSwinConfig:
    image_size: int = 224
    patch_size: int = 4
    in_channels: int = 3
    num_classes: int = 100
    embed_dim: int = 48
    depths: tuple[int, int, int, int] = (2, 2, 2, 2)
    num_heads: tuple[int, int, int, int] = (3, 6, 12, 24)
    window_size: int = 7
    dropout: float = 0.0


class ScratchSwinTransformer(nn.Module):
    """Compact Swin classifier with hierarchical shifted-window attention."""

    def __init__(self, config: ScratchSwinConfig = ScratchSwinConfig()):
        super().__init__()
        self.config = config
        self.patch_embed = PatchEmbed(
            config.image_size,
            config.patch_size,
            config.in_channels,
            config.embed_dim,
        )
        resolution = config.image_size // config.patch_size
        layers = []
        dim = config.embed_dim
        for stage_idx, depth in enumerate(config.depths):
            input_resolution = (resolution, resolution)
            layers.append(
                BasicLayer(
                    dim=dim,
                    input_resolution=input_resolution,
                    depth=depth,
                    num_heads=config.num_heads[stage_idx],
                    window_size=config.window_size,
                    dropout=config.dropout,
                    downsample=stage_idx < len(config.depths) - 1,
                )
            )
            if stage_idx < len(config.depths) - 1:
                resolution //= 2
                dim *= 2
        self.layers = nn.ModuleList(layers)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, config.num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x, height, width = self.patch_embed(x)
        for layer in self.layers:
            x, height, width = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.head(x)


def run_swin_shape_tests() -> dict[str, tuple[int, ...] | bool]:
    """Small synthetic checks for required scratch Swin tensor operations."""

    sample = torch.randn(2, 56, 56, 3)
    windows = window_partition(sample, 7)
    restored = window_reverse(windows, 7, 56, 56)
    if not torch.allclose(sample, restored):
        raise AssertionError("bad window_reverse")

    merging = PatchMerging((56, 56), 48)
    merged, h, w = merging(torch.randn(2, 56 * 56, 48))
    if tuple(merged.shape) != (2, 28 * 28, 96):
        raise AssertionError("bad patch_merging")

    block = SwinBlock(48, (56, 56), num_heads=3, window_size=7, shift_size=3)
    if block.attn_mask is None:
        raise AssertionError("bad shift_mask")
    block_out = block(torch.randn(2, 56 * 56, 48))

    model = ScratchSwinTransformer(
        ScratchSwinConfig(image_size=224, embed_dim=24, num_heads=(3, 6, 12, 24))
    )
    logits = model(torch.randn(2, 3, 224, 224))
    if tuple(logits.shape) != (2, 100):
        raise AssertionError("bad logits")
    return {
        "window_partition": tuple(windows.shape),
        "window_reverse_matches": True,
        "patch_merging": tuple(merged.shape),
        "shifted_block": tuple(block_out.shape),
        "end_to_end_logits": tuple(logits.shape),
    }


Import the remaining Problem 2 tools, set the seed, select the device, and define the required hyperparameters.


In [4]:
import json

import torch
from transformers import AutoImageProcessor, SwinForImageClassification

WORK_DIR, DATA_ROOT, RESULT_DIR = project_paths("problem2")

SEED = 42
set_seed(SEED)
device_info = select_device()
device = device_info.device
print("Selected device:", device_info.as_dict())

EPOCHS = 5
BATCH_SIZE = 32
PRETRAINED_LR = 2e-5
SCRATCH_LR = 0.001
IMAGE_SIZE = 224
ensure_dir(RESULT_DIR / "checkpoints")
ensure_dir(RESULT_DIR / "plots")


Mounted at /content/drive
Selected device: {'device': 'cuda', 'name': 'Tesla T4', 'cuda_available': True, 'cuda_device_count': 1}


PosixPath('/content/drive/MyDrive/homework_5/results/problem2/plots')

Load the Hugging Face Swin image processor and prepare CIFAR-100 with the same 224x224 preprocessing.


In [5]:
PRETRAINED_CHECKPOINTS = {
    "swin_tiny_pretrained_frozen": "microsoft/swin-tiny-patch4-window7-224",
    "swin_small_pretrained_frozen": "microsoft/swin-small-patch4-window7-224",
}

processor = AutoImageProcessor.from_pretrained(PRETRAINED_CHECKPOINTS["swin_tiny_pretrained_frozen"])
mean = tuple(processor.image_mean)
std = tuple(processor.image_std)

loaders = get_cifar100_dataloaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    mean=mean,
    std=std,
    augment_train=True,
    num_workers=2,
    seed=SEED,
    download=True,
)
images, labels = next(iter(loaders["train"]))
print("Swin training batch:", images.shape, labels.min().item(), labels.max().item())


Swin training batch: torch.Size([32, 3, 224, 224]) 2 94


Freeze each pretrained Swin backbone and verify that only the classifier parameters are trainable.


In [6]:
def freeze_swin_backbone_train_classifier(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True
    trainable_names = [name for name, param in model.named_parameters() if param.requires_grad]
    only_classifier = all(name.startswith("classifier.") for name in trainable_names)
    if not only_classifier:
        raise AssertionError("bad freeze")
    return trainable_names, only_classifier


def build_pretrained_swin(checkpoint):
    model = SwinForImageClassification.from_pretrained(
        checkpoint,
        num_labels=100,
        ignore_mismatched_sizes=True,
    )
    trainable_names, only_classifier = freeze_swin_backbone_train_classifier(model)
    print(checkpoint, "trainable parameters:", trainable_names)
    print("Only classifier trainable:", only_classifier)
    return model, trainable_names, only_classifier


def no_frozen_backbone_gradients(model):
    bad = []
    for name, param in model.named_parameters():
        if not name.startswith("classifier.") and param.grad is not None:
            if torch.any(param.grad != 0):
                bad.append(name)
    return bad


Run scratch Swin checks for window partitioning, window reversal, shifted-window masks, patch merging, and logits shape.


In [7]:
print("Scratch Swin shape tests:")
shape_tests = run_swin_shape_tests()
shape_tests


Scratch Swin shape tests:


{'window_partition': (128, 7, 7, 3),
 'window_reverse_matches': True,
 'patch_merging': (2, 784, 96),
 'shifted_block': (2, 3136, 48),
 'end_to_end_logits': (2, 100)}

Define the shared Swin experiment runner for output checks, optimizer checks, complexity estimates, training, and exports.


In [8]:
def execute_swin_experiment(model_name, model, lr, pretrained, frozen_backbone, trainable_group_names, config):
    model = model.to(device)
    dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    output = model(dummy)
    logits = output.logits if hasattr(output, "logits") else output
    assert tuple(logits.shape) == (1, 100), "bad logits"

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=lr)
    verify_optimizer_parameters(model, optimizer)

    total_params, trainable_params = count_parameters(model)
    complexity = estimate_macs_thop(model, (1, 3, IMAGE_SIZE, IMAGE_SIZE), device)
    checkpoint_path = RESULT_DIR / "checkpoints" / f"{model_name}.pt"

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    history, fit_summary = fit_classifier(
        model=model,
        train_loader=loaders["train"],
        eval_loader=loaders["test"],
        optimizer=optimizer,
        device=device,
        epochs=EPOCHS,
        checkpoint_path=checkpoint_path,
        frozen_eval_modules=("swin",) if frozen_backbone else (),
    )

    frozen_grad_violations = no_frozen_backbone_gradients(model) if frozen_backbone else []
    if frozen_grad_violations:
        raise AssertionError("bad frozen_grad")

    peak_memory_mb = None
    if device.type == "cuda":
        peak_memory_mb = torch.cuda.max_memory_allocated(device) / (1024**2)

    history_path = RESULT_DIR / f"training_history_{model_name}.csv"
    plot_path = RESULT_DIR / "plots" / f"{model_name}_training_curves.png"
    save_training_history(history, history_path)
    plot_training_history(history, model_name, plot_path)

    row = {
        "model_name": model_name,
        "configuration": json.dumps(config),
        "total_parameters": total_params,
        "trainable_parameters": trainable_params,
        "model_size_mb": model_size_megabytes(model),
        "macs_per_forward": complexity.get("macs"),
        "estimated_flops_if_2x_macs": complexity.get("estimated_flops_if_2x_macs"),
        "complexity_tool": complexity.get("complexity_tool"),
        "complexity_status": complexity.get("complexity_status"),
        "flops_convention": complexity.get("flops_convention"),
        "total_training_time_seconds": fit_summary["total_training_time_seconds"],
        "mean_epoch_time_seconds": fit_summary["mean_epoch_time_seconds"],
        "test_accuracy": fit_summary["final_eval_accuracy"],
        "batch_size": BATCH_SIZE,
        "learning_rate": lr,
        "epochs": EPOCHS,
        "optimizer": "Adam",
        "random_seed": SEED,
        "device": str(device),
        "device_name": device_info.name,
        "pretrained_weights_used": pretrained,
        "frozen_backbone": frozen_backbone,
        "trainable_parameter_groups": json.dumps(trainable_group_names),
        "input_resolution": IMAGE_SIZE,
        "preprocessing": "Hugging Face Swin image processor mean/std with Resize(224)",
        "peak_gpu_memory_mb": peak_memory_mb,
        "checkpoint_path": str(checkpoint_path),
        "history_path": str(history_path),
        "plot_path": str(plot_path),
    }
    return row, {"history": history, "summary": fit_summary, "complexity": complexity}


Train Swin-Tiny and Swin-Small with frozen pretrained backbones, then train the scratch Swin model.


In [9]:
all_results = []
detail_results = {"scratch_shape_tests": shape_tests}

for model_name, checkpoint in PRETRAINED_CHECKPOINTS.items():
    print(f"Running {model_name}")
    model, trainable_names, only_classifier = build_pretrained_swin(checkpoint)
    row, details = execute_swin_experiment(
        model_name=model_name,
        model=model,
        lr=PRETRAINED_LR,
        pretrained=True,
        frozen_backbone=True,
        trainable_group_names=trainable_names,
        config={
            "checkpoint": checkpoint,
            "num_labels": 100,
            "ignore_mismatched_sizes": True,
            "only_classifier_trainable": only_classifier,
        },
    )
    all_results.append(row)
    detail_results[model_name] = details

print("Running scratch Swin Transformer")
scratch_config = ScratchSwinConfig(image_size=IMAGE_SIZE)
scratch_model = ScratchSwinTransformer(scratch_config)
row, details = execute_swin_experiment(
    model_name="swin_scratch_compact",
    model=scratch_model,
    lr=SCRATCH_LR,
    pretrained=False,
    frozen_backbone=False,
    trainable_group_names=["all"],
    config=scratch_config.__dict__,
)
all_results.append(row)
detail_results["swin_scratch_compact"] = details


Running swin_tiny_pretrained_frozen


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.
[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


microsoft/swin-tiny-patch4-window7-224 trainable parameters: ['classifier.weight', 'classifier.bias']
Only classifier trainable: True
Running swin_small_pretrained_frozen


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.
[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-small-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


microsoft/swin-small-patch4-window7-224 trainable parameters: ['classifier.weight', 'classifier.bias']
Only classifier trainable: True
Running scratch Swin Transformer


Save the Problem 2 CSV and JSON outputs.


In [10]:
save_csv(all_results, RESULT_DIR / "problem2_results.csv")
save_json({"results": all_results, "details": detail_results}, RESULT_DIR / "problem2_results.json")
print("Saved Problem 2 artifacts to:", RESULT_DIR)
all_results


Saved Problem 2 artifacts to: /content/drive/MyDrive/homework_5/results/problem2


[{'model_name': 'swin_tiny_pretrained_frozen',
  'configuration': '{"checkpoint": "microsoft/swin-tiny-patch4-window7-224", "num_labels": 100, "ignore_mismatched_sizes": true, "only_classifier_trainable": true}',
  'total_parameters': 27596254,
  'trainable_parameters': 76900,
  'model_size_mb': 105.27135467529297,
  'macs_per_forward': 4364674560,
  'estimated_flops_if_2x_macs': 8729349120,
  'complexity_tool': 'thop',
  'complexity_status': 'ok',
  'flops_convention': 'thop returns MACs; FLOPs are often approximated as 2x MACs',
  'total_training_time_seconds': 1386.0040750649996,
  'mean_epoch_time_seconds': 230.08543603439995,
  'test_accuracy': 0.6642,
  'batch_size': 32,
  'learning_rate': 2e-05,
  'epochs': 5,
  'optimizer': 'Adam',
  'random_seed': 42,
  'device': 'cuda',
  'device_name': 'Tesla T4',
  'pretrained_weights_used': True,
  'frozen_backbone': True,
  'trainable_parameter_groups': '["classifier.weight", "classifier.bias"]',
  'input_resolution': 224,
  'preprocessin

Create a downloadable zip file and list the saved Problem 2 artifacts.


In [11]:
import shutil

for path in sorted(RESULT_DIR.rglob("*")):
    if path.is_file():
        print(path)

zip_path = shutil.make_archive(str(RESULT_DIR), "zip", RESULT_DIR)
print("Results zip:", zip_path)


/content/drive/MyDrive/homework_5/results/problem2/checkpoints/swin_scratch_compact.pt
/content/drive/MyDrive/homework_5/results/problem2/checkpoints/swin_small_pretrained_frozen.pt
/content/drive/MyDrive/homework_5/results/problem2/checkpoints/swin_tiny_pretrained_frozen.pt
/content/drive/MyDrive/homework_5/results/problem2/plots/swin_scratch_compact_training_curves.png
/content/drive/MyDrive/homework_5/results/problem2/plots/swin_small_pretrained_frozen_training_curves.png
/content/drive/MyDrive/homework_5/results/problem2/plots/swin_tiny_pretrained_frozen_training_curves.png
/content/drive/MyDrive/homework_5/results/problem2/problem2_results.csv
/content/drive/MyDrive/homework_5/results/problem2/problem2_results.json
/content/drive/MyDrive/homework_5/results/problem2/training_history_swin_scratch_compact.csv
/content/drive/MyDrive/homework_5/results/problem2/training_history_swin_small_pretrained_frozen.csv
/content/drive/MyDrive/homework_5/results/problem2/training_history_swin_tin